###DimUsers

In [0]:

df=spark.read.format('parquet').load('abfss://bronze@storagekalyan56.dfs.core.windows.net/DimUser')

In [0]:
display(df)

In [0]:
df_user=spark.readStream.format('cloudFiles')\
                        .option('cloudFiles.format','parquet')\
                        .option('cloudFiles.schemaLocation','abfss://silver@storagekalyan56.dfs.core.windows.net/DimUser/checkpoint')\
                        .load('abfss://bronze@storagekalyan56.dfs.core.windows.net/DimUser')

In [0]:
display(df_user)

In [0]:
from pyspark.sql.functions import *

In [0]:
df_user=df_user.withColumn('user_name',upper(col("user_name")))
df_user.display()

In [0]:
import os
print(os.path.join(os.getcwd(),'..','..'))

In [0]:
import os
project_path = os.path.join(os.getcwd(), '..', '..')
import sys
sys.path.append(project_path)

import importlib
for mod in [k for k in sys.modules if k == 'utils' or k.startswith('utils.')]:
    del sys.modules[mod]
importlib.invalidate_caches()

from utils.transformations import kalyan_var
from utils.transformations import reusable



In [0]:
df_user_obj=reusable()
df_user=df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user=df_user.dropDuplicates(['user_id'])
df_user.display()

In [0]:
df_user.writeStream.format('delta')\
.outputMode('append')\
.option('checkpointLocation','abfss://silver@storagekalyan56.dfs.core.windows.net/DimUser/checkpoint')\
.trigger(once=True)\
.option('path','abfss://silver@storagekalyan56.dfs.core.windows.net/DimUser/data')\
.toTable('spotify_cata.silver.DimUser')